In [10]:
# ─────────────────────────────────────────
# CELL 1: DATA GENERATION
# ─────────────────────────────────────────
import numpy as np
import pandas as pd

np.random.seed(42)
N = 3000

temperature = np.random.normal(18, 7, N)
humidity    = np.random.normal(60, 15, N)
wind_speed  = np.abs(np.random.normal(4, 2, N))
visibility  = np.random.normal(38, 12, N)
dew_point   = temperature - np.random.uniform(2, 8, N)

noise  = np.random.normal(0, 80, N)
energy = (
    200
    + 3.5 * temperature**2
    - 0.8 * humidity * temperature
    + 20  * wind_speed
    - 1.2 * visibility
    + 15  * dew_point
    + noise
)

df = pd.DataFrame({
    "temperature" : temperature,
    "humidity"    : humidity,
    "wind_speed"  : wind_speed,
    "visibility"  : visibility,
    "dew_point"   : dew_point,
    "energy"      : energy
})

print(f"Dataset shape: {df.shape}")
print(f"Rows: {len(df)}  |  Features: 5  |  Target: energy\n")
df.head()

Dataset shape: (3000, 6)
Rows: 3000  |  Features: 5  |  Target: energy



,temperature,humidity,wind_speed,visibility,dew_point,energy
0,21.476999,31.382887,1.771837,47.184826,17.222230,1443.677358
1,17.032150,47.094225,2.738138,50.880952,11.566202,699.603886
2,22.533820,53.795917,2.115880,43.984279,17.901868,1229.929729
3,28.661209,88.315315,2.904008,14.690028,26.557133,1421.280289
4,16.360926,68.348297,3.571699,36.134931,8.396940,256.971571


In [11]:
# ─────────────────────────────────────────
# CELL 3: PREPROCESSING
# ─────────────────────────────────────────
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# STEP 1: Separate features (X) and target (y)
# X = what we feed into the model (5 weather columns)
# y = what we want to predict (energy)
X = df[["temperature", "humidity", "wind_speed", "visibility", "dew_point"]].values
y = df["energy"].values.reshape(-1, 1)   # reshape to column vector

print("X shape:", X.shape)   # (3000, 5)
print("y shape:", y.shape)   # (3000, 1)

# STEP 2: Scale the data
# Neural networks train better when all values are in similar range
# StandardScaler converts everything to mean=0 and std=1
scaler_X = StandardScaler()
scaler_y = StandardScaler()

X_scaled = scaler_X.fit_transform(X)   # fit AND transform on full data
y_scaled = scaler_y.fit_transform(y)


# STEP 3: Split into Train and Test sets
# 80% for training, 20% for testing
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y_scaled,
    test_size=0.2,       # 20% goes to test
    random_state=42      # fixed seed so split is always same
)

print("\nTraining samples :", len(X_train))   # 2400
print("Testing  samples :", len(X_test))    # 600

X shape: (3000, 5)
y shape: (3000, 1)

Training samples : 2400
Testing  samples : 600


## Step 4 — Build & Train Model (Scikit-learn)

Before jumping into PyTorch, we first build the same neural network using **MLPRegressor** from scikit-learn.
Same architecture, same data — just the simple version first so we understand the concept clearly.
We use **SGD optimizer** here, then later switch to PyTorch where we compare SGD vs Adam properly.

In [17]:
# ─────────────────────────────────────────
# CELL 4: BUILD MODEL — using MLPRegressor
# (MLP = Multi Layer Perceptron)
# No PyTorch, pure scikit-learn!
# ─────────────────────────────────────────
from sklearn.neural_network import MLPRegressor

# STEP 1: Define the model
# hidden_layer_sizes = (64, 128, 64, 32)
# this means:
#   Layer 1 → 64  neurons
#   Layer 2 → 128 neurons
#   Layer 3 → 64  neurons
#   Layer 4 → 32  neurons
# input(5) and output(1) are handled automatically by sklearn

model_sgd = MLPRegressor(
    hidden_layer_sizes = (64, 128, 64, 32),  # 4 hidden layers
    activation         = "relu",             # ReLU activation in each layer
    solver             = "sgd",              # SGD optimizer
    learning_rate_init = 0.01,               # starting learning rate
    momentum           = 0.9,                # SGD momentum
    max_iter           = 30,             # number of epochs
    batch_size         = len(X_train),
   # random_state       = 42,                 # reproducibility
    verbose            = True                # print loss every 100 epochs
)
# STEP 2: Train the model
# just one line — fit on training data
model_sgd.fit(X_train, y_train.ravel())  # .ravel() flattens y to 1D

# STEP 3: Quick check
print("\n✓ Model trained!")
print(f"  Layers     : Input(5) → 64 → 128 → 64 → 32 → Output(1)")
print(f"  Optimizer  : SGD")
print(f"  Iterations : {model_sgd.n_iter_}")
print(f"  Final Loss : {model_sgd.loss_:.5f}")

Iteration 1, loss = 0.48112340
Iteration 2, loss = 0.47094494
Iteration 3, loss = 0.45685531
Iteration 4, loss = 0.43962902
Iteration 5, loss = 0.41992250
Iteration 6, loss = 0.39813767
Iteration 7, loss = 0.37475425
Iteration 8, loss = 0.35016619
Iteration 9, loss = 0.32476690
Iteration 10, loss = 0.29877629
Iteration 11, loss = 0.27223138
Iteration 12, loss = 0.24541037
Iteration 13, loss = 0.21873379
Iteration 14, loss = 0.19239765
Iteration 15, loss = 0.16676839
Iteration 16, loss = 0.14236783
Iteration 17, loss = 0.11992434
Iteration 18, loss = 0.10001741
Iteration 19, loss = 0.08306540
Iteration 20, loss = 0.06929795
Iteration 21, loss = 0.05875145
Iteration 22, loss = 0.05127774
Iteration 23, loss = 0.04647775
Iteration 24, loss = 0.04377982
Iteration 25, loss = 0.04254714
Iteration 26, loss = 0.04212340
Iteration 27, loss = 0.04192631
Iteration 28, loss = 0.04150477
Iteration 29, loss = 0.04058805
Iteration 30, loss = 0.03907576

✓ Model trained!
  Layers     : Input(5) → 64 → 

c:\Users\Zalaid\anaconda3\envs\llms\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (30) reached and the optimization hasn't converged yet.
  warnings.warn(


# Why We Move from MLPRegressor → PyTorch

We just trained our model using **scikit-learn's MLPRegressor**.
It worked, but it comes with serious limitations for real deep learning work.

---

## Limitations of MLPRegressor

### 1. No GPU Support
- MLPRegressor runs only on **CPU**
- For large datasets, training becomes very slow
- PyTorch can run on **GPU**, making it 10x-100x faster

### 2. No Custom Control
- You cannot control the **training loop** step by step
- You cannot add custom logic between layers
- PyTorch lets you control **every single step**

### 3. Limited Architecture
- MLPRegressor only builds simple fully connected networks
- You cannot build CNNs, RNNs, Transformers, or anything advanced
- PyTorch can build **any architecture ever invented**

### 4. No Batch Training
- MLPRegressor loads **all data at once** into memory
- For huge datasets this will **crash your RAM**
- PyTorch uses **DataLoaders** to feed data in small batches

### 5. Cannot Save/Load Properly
- MLPRegressor saving is basic (just pickle)
- PyTorch has a proper **model checkpointing** system
- You can save mid-training and resume anytime

### 6. No Flexibility in Loss Functions
- MLPRegressor has fixed loss functions only
- PyTorch lets you write **custom loss functions** for any problem

---

## Conclusion

> MLPRegressor is great for **quick experiments and learning concepts.**
> But for real, production-level deep learning — **PyTorch is the industry standard.**

In the next cells we rebuild the **exact same model in PyTorch**
and you will see the difference in control, visibility, and performance.

#  Now We Train in PyTorch — Adam Optimizer


We already saw how **MLPRegressor** trains a neural network the easy way.  
But it had limitations — no GPU, no control, not used in real world.

Now we **rebuild the exact same model in PyTorch** — the framework used by:
-  OpenAI (ChatGPT)
-  Stability AI (Image Generation)
-  Meta AI (Research)
-  Tesla (Autopilot)


## Why Adam?

> Adam is a **smarter optimizer** — it automatically adjusts the learning rate  
> for every single neuron individually, leading to **faster and better convergence.**

Instead of one fixed learning rate for everyone (SGD),  
Adam gives **each weight its own personalized learning rate.** 

---

## Same Model. Same Data. Better Optimizer. Let's See the Difference. 

In [4]:
# ─────────────────────────────────────────
# CELL: PYTORCH MODEL — ADAM
# ─────────────────────────────────────────
import torch
import torch.nn as nn

# STEP 1: Convert data to PyTorch tensors
X_train_t = torch.FloatTensor(X_train)
y_train_t = torch.FloatTensor(y_train)
X_test_t  = torch.FloatTensor(X_test)
y_test_t  = torch.FloatTensor(y_test)

# STEP 2: Build the model (same architecture)
class EnergyNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(5,   64),  nn.ReLU(),   # layer 1
            nn.Linear(64, 128),  nn.ReLU(),   # layer 2
            nn.Linear(128, 64),  nn.ReLU(),   # layer 3
            nn.Linear(64,  32),  nn.ReLU(),   # layer 4
            nn.Linear(32,   1)                # output
        )

    def forward(self, x):
        return self.model(x)

model = EnergyNet()
print(model)

# STEP 3: Loss and optimizer
# only change from SGD version — optimizer is now Adam
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)  # Adam!

# STEP 4: Training loop
EPOCHS = 30
losses = []

for epoch in range(EPOCHS):
    model.train()
    optimizer.zero_grad()              # clear old gradients
    pred = model(X_train_t)            # forward pass
    loss = criterion(pred, y_train_t)  # calculate loss
    loss.backward()                    # backpropagation
    optimizer.step()                   # update weights

    losses.append(loss.item())

    if (epoch + 1) % 1 == 0:
        print(f"Epoch {epoch+1}/{EPOCHS}  Loss: {loss.item():.5f}")

print("\n✅ Adam training complete!")

EnergyNet(
  (model): Sequential(
    (0): Linear(in_features=5, out_features=64, bias=True)
    (1): ReLU()
    (2): Linear(in_features=64, out_features=128, bias=True)
    (3): ReLU()
    (4): Linear(in_features=128, out_features=64, bias=True)
    (5): ReLU()
    (6): Linear(in_features=64, out_features=32, bias=True)
    (7): ReLU()
    (8): Linear(in_features=32, out_features=1, bias=True)
  )
)
Epoch 1/30  Loss: 0.98586
Epoch 2/30  Loss: 0.96442
Epoch 3/30  Loss: 0.94361
Epoch 4/30  Loss: 0.92203
Epoch 5/30  Loss: 0.89870
Epoch 6/30  Loss: 0.87307
Epoch 7/30  Loss: 0.84489
Epoch 8/30  Loss: 0.81425
Epoch 9/30  Loss: 0.78136
Epoch 10/30  Loss: 0.74626
Epoch 11/30  Loss: 0.70900
Epoch 12/30  Loss: 0.66959
Epoch 13/30  Loss: 0.62813
Epoch 14/30  Loss: 0.58494
Epoch 15/30  Loss: 0.54040
Epoch 16/30  Loss: 0.49524
Epoch 17/30  Loss: 0.45044
Epoch 18/30  Loss: 0.40721
Epoch 19/30  Loss: 0.36693
Epoch 20/30  Loss: 0.33081
Epoch 21/30  Loss: 0.29946
Epoch 22/30  Loss: 0.27232
Epoch 23/30

#  Same Model — Now With SGD Optimizer in Pytorch

Let's swap Adam → SGD and train again.  
Watch the loss carefully — **who converges faster and hits lower loss wins.**

In [5]:
# RESET weights so SGD starts fresh
model_sgd = EnergyNet()

optimizer = torch.optim.SGD(model_sgd.parameters(), lr=0.01, momentum=0.9)

EPOCHS = 30
losses_sgd = []

for epoch in range(EPOCHS):
    model_sgd.train()
    optimizer.zero_grad()
    pred = model_sgd(X_train_t)
    loss = criterion(pred, y_train_t)
    loss.backward()
    optimizer.step()
    losses_sgd.append(loss.item())

    if (epoch + 1) % 1 == 0:
        print(f"Epoch {epoch+1}/{EPOCHS}  Loss: {loss.item():.5f}")

print("\n SGD training complete!")

Epoch 1/30  Loss: 0.98817
Epoch 2/30  Loss: 0.98660
Epoch 3/30  Loss: 0.98369
Epoch 4/30  Loss: 0.97969
Epoch 5/30  Loss: 0.97486
Epoch 6/30  Loss: 0.96939
Epoch 7/30  Loss: 0.96347
Epoch 8/30  Loss: 0.95723
Epoch 9/30  Loss: 0.95079
Epoch 10/30  Loss: 0.94425
Epoch 11/30  Loss: 0.93770
Epoch 12/30  Loss: 0.93088
Epoch 13/30  Loss: 0.92348
Epoch 14/30  Loss: 0.91520
Epoch 15/30  Loss: 0.90581
Epoch 16/30  Loss: 0.89512
Epoch 17/30  Loss: 0.88293
Epoch 18/30  Loss: 0.86910
Epoch 19/30  Loss: 0.85347
Epoch 20/30  Loss: 0.83587
Epoch 21/30  Loss: 0.81610
Epoch 22/30  Loss: 0.79389
Epoch 23/30  Loss: 0.76891
Epoch 24/30  Loss: 0.74077
Epoch 25/30  Loss: 0.70914
Epoch 26/30  Loss: 0.67368
Epoch 27/30  Loss: 0.63404
Epoch 28/30  Loss: 0.59001
Epoch 29/30  Loss: 0.54161
Epoch 30/30  Loss: 0.48904

 SGD training complete!
